In [ ]:
import pathlib, glob, os, re, csv, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.backends.backend_pdf as mpdf
from astropy.io import fits
from collections import defaultdict

# ── paths ─────────────────────────────────────────────────────────────────────
BASE         = "kmos_ob_proc_server"
SUFFIX       = "_s_i_bkg2d_rv_tell.fits"
COMB_BASE    = "kmos_proc_server_comb"
REDSHIFT_CSV = pathlib.Path("scripts") / "target_redshifts.csv"
ECDFS_TABLE  = pathlib.Path("scripts") / "table_ifus_ECDFS 2.fits"
COSMOS_TABLE = pathlib.Path("scripts") / "table_ifus_COSMOS_Field_exptime.fits"

# ── extraction ─────────────────────────────────────────────────────────────────
APERTURE          = 5       # px box aperture
EDGE_MARGIN       = 4       # px edge erosion (same as stacked_showcase)
MAX_CENTER_OFFSET = 4.0     # px; farther peaks → IFU centre
HA_REST           = 0.65628 # Hα rest wavelength µm
OIII_REST         = 0.50069 # [OIII]λ5007 rest wavelength µm
HB_REST           = 0.48612 # Hβ rest wavelength µm
SNR_WIN_HALF      = 0.0025  # ±µm signal window around line
CONT_HALF         = 0.015   # ±µm continuum window (blueward of line)
ZOOM_HALF         = 0.035   # ±µm for line-zoom panel

# ── output ────────────────────────────────────────────────────────────────────
OUT_PDF = "kmos_comb_compare.pdf"


In [ ]:
# ── robust line S/N: scripts/line_snr.py ─────────────────────────────────────
import sys, importlib, pathlib
_SCRIPTS = str(pathlib.Path("scripts").resolve())
if _SCRIPTS not in sys.path:
    sys.path.insert(0, _SCRIPTS)
import line_snr as _lsnr
importlib.reload(_lsnr)

# ── redshift catalogue ────────────────────────────────────────────────────────
_Z_TABLE   = {}
_Z_TYPE    = {}
_Z_QUALITY = {}

with open(REDSHIFT_CSV, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        k = (row["field"].strip(), str(row["arm_target_id"]).strip())
        try:
            _Z_TABLE[k] = float(row["redshift"])
        except (ValueError, KeyError):
            pass
        _Z_TYPE[k] = row.get("redshift_type", "").strip()
        try:
            _Z_QUALITY[k] = int(float(row.get("quality", "nan")))
        except (ValueError, TypeError):
            _Z_QUALITY[k] = None

# ── OB folder name → dp_id (colons restored) ──────────────────────────────────
def ob_to_dpid(ob_name):
    if "T" not in ob_name:
        return ob_name
    date_part, time_part = ob_name.split("T", 1)
    parts = time_part.split("-", 2)
    if len(parts) == 3:
        return f"{date_part}T{parts[0]}:{parts[1]}:{parts[2]}"
    return ob_name

# ── wavelength axis from FITS header ──────────────────────────────────────────
def wavelength_axis(hdr):
    n   = hdr["NAXIS3"]
    crv = hdr["CRVAL3"]
    crp = hdr.get("CRPIX3", 1.0)
    dlt = hdr.get("CDELT3") or hdr.get("CD3_3")
    return crv + (np.arange(n) - (crp - 1)) * dlt

# ── active IFUs (truncation-safe) ─────────────────────────────────────────────
def active_ifus(hdul):
    result = []
    for i in range(1, 25):
        try:
            d = hdul[i].data
            if d is not None and np.ndim(d) == 3:
                result.append(i)
        except Exception:
            pass
    return result

# ── IFU geometric centre ──────────────────────────────────────────────────────
def ifu_center(cube):
    mask = np.isfinite(cube).any(axis=0)
    if not mask.any():
        return cube.shape[1] // 2, cube.shape[2] // 2
    ys, xs = np.where(mask)
    return int(round(ys.mean())), int(round(xs.mean()))

# ── aperture extraction from a 3-D cube array ─────────────────────────────────
def aperture_from_cube(cube, center, box=APERTURE, margin=EDGE_MARGIN):
    py, px = center
    h2 = box // 2
    y0, y1 = max(py - h2, 0), min(py + h2 + 1, cube.shape[1])
    x0, x1 = max(px - h2, 0), min(px + h2 + 1, cube.shape[2])
    patch = cube[:, y0:y1, x0:x1].reshape(cube.shape[0], -1)
    valid = np.isfinite(patch)
    if margin > 0:
        valid[:, :margin]  = False; valid[:, -margin:] = False
        valid[:margin, :]  = False; valid[-margin:, :] = False
    cleaned = np.where(valid, patch, np.nan)
    spec = np.full(cube.shape[0], np.nan)
    ok = valid.any(axis=1)
    spec[ok] = np.nansum(cleaned[ok], axis=1)
    return spec

# ── band wavelength coverage ───────────────────────────────────────────────────
BAND_RANGES = {
    "IZ": (0.780, 1.090), "YJ": (1.000, 1.359),
    "HK": (1.456, 2.450), "H":  (1.456, 1.850), "K": (1.934, 2.450),
}

# ── pick best visible emission line for this band/redshift ────────────────────
def pick_line(band, z):
    """Return (rest_wl_um, line_name, obs_wl_um) for Ha, then [OIII], then Hb.
    Falls back to Ha if nothing is in band (will produce no-data spatial maps)."""
    wmin, wmax = BAND_RANGES.get(band, (0.5, 2.5))
    for rest_wl, name in [(HA_REST, "Ha"), (OIII_REST, "OIII"), (HB_REST, "Hb")]:
        obs_wl = rest_wl * (1 + z)
        if wmin + SNR_WIN_HALF < obs_wl < wmax - SNR_WIN_HALF:
            return rest_wl, name, obs_wl
    return HA_REST, "Ha", HA_REST * (1 + z)  # fallback

# ── generic line SNR, peak centre, and collapsed image ────────────────────────
def line_snr(wave, spec, obs_wl):
    """Robust S/N via scripts/line_snr.py: Gaussian-fit line flux / error
    (spikes masked, LSF/velocity priors, correlated noise; see that file)."""
    # obs_wl is already redshifted -> z=0, no automatic companions
    return _lsnr.line_snr(wave, spec, 0.0, obs_wl, companions=(),
                          win_half=SNR_WIN_HALF)

def line_center(cube, wave, obs_wl):
    mask = np.abs(wave - obs_wl) <= SNR_WIN_HALF
    if not mask.any():
        return None
    img = np.nansum(cube[mask], axis=0)
    if not np.isfinite(img).any():
        return None
    idx  = np.unravel_index(np.nanargmax(img), img.shape)
    geom = ifu_center(cube)
    if np.hypot(idx[0] - geom[0], idx[1] - geom[1]) > MAX_CENTER_OFFSET:
        return geom
    return idx

def line_image(cube, wave, obs_wl, half=0.010):
    m = np.abs(wave - obs_wl) <= half
    if not m.any():
        return None
    return np.nansum(cube[m], axis=0)

print("Helpers loaded.")


In [ ]:
# ── dpid × ifu → group_id from FITS catalogue tables ─────────────────────────
dpid_ifu_to_group = {}

for tbl_path in [ECDFS_TABLE, COSMOS_TABLE]:
    if not tbl_path.exists():
        print(f"  Warning: {tbl_path} not found")
        continue
    with fits.open(tbl_path) as hdul:
        t = hdul[1].data
        for i in range(len(t)):
            dpid_ifu_to_group[(t["dp_id"][i].strip(), int(t["ifu"][i]))] = int(t["group_id"][i])

# ── synthetic group IDs for fields not in catalogue ───────────────────────────
_field_tid_to_group = {}
_synthetic_next     = 90000

def ensure_group(field, tid):
    global _synthetic_next
    k = (field, tid)
    if k not in _field_tid_to_group:
        _field_tid_to_group[k] = _synthetic_next
        _synthetic_next += 1
    return _field_tid_to_group[k]

group_to_meta = {}  # group_id → (field, tid, z, z_type, z_quality)

print(f"Catalogue: {len(dpid_ifu_to_group)} dp_id×ifu entries loaded.")


In [ ]:
# ── scan kmos_proc_server_comb → {(field, tid): {path, band, exptime}} ────────
ext_index = {}

for fits_path in sorted(glob.glob(
        os.path.join(COMB_BASE, "**", "DATACUBE_COMBINED_*.fits"), recursive=True)):
    fname = os.path.basename(fits_path)
    # DATACUBE_COMBINED_<field>_<tid>_<band>.fits
    m = re.match(r"DATACUBE_COMBINED_(.+)_([^_]+)_(IZ|YJ|HK|H|K)\.fits$", fname)
    if not m:
        continue
    field, tid, band = m.group(1), m.group(2), m.group(3)
    with fits.open(fits_path) as hdul:
        exptime = hdul[0].header.get("EXPTIME", np.nan)
    ext_index[(field, tid)] = {"path": fits_path, "band": band, "exptime": exptime}

print(f"External index: {len(ext_index)} cubes across "
      f"{len(set(k[0] for k in ext_index))} fields")
for field in sorted(set(k[0] for k in ext_index)):
    n     = sum(1 for k in ext_index if k[0] == field)
    bands = sorted(set(ext_index[k]["band"] for k in ext_index if k[0] == field))
    expt  = np.unique([ext_index[k]["exptime"] for k in ext_index if k[0] == field])
    print(f"  {field}: {n} cubes, band={bands}, exptime={expt}")


In [ ]:
# ── stack user's per-OB cubes (mirrors kmos_stacked_showcase logic) ───────────
group_data = defaultdict(lambda: {
    "spectra": [], "wave": None,
    "best_cube": None, "best_center": None, "best_snr": -np.inf,
    "obs": set(),
})

n_matched = n_skipped = 0

for path in sorted(glob.glob(os.path.join(BASE, "*", "science", f"*{SUFFIX}"))):
    parts  = path.replace("\\", "/").split("/")
    ob     = parts[-3]
    dpid   = ob_to_dpid(ob)

    with fits.open(path) as hdul:
        ifus  = active_ifus(hdul)
        hdr0  = hdul[0].header
        band  = (hdr0.get("HIERARCH ESO INS GRAT1 ID")
                 or hdr0.get("ESO INS GRAT1 ID", "??"))
        field_hdr = (hdr0.get("ESO OBS TARG NAME") or "").strip()

        for ifu in ifus:
            # resolve group_id
            group_id = dpid_ifu_to_group.get((dpid, ifu))
            if group_id is None:
                key_hdr = f"ESO OCS ARM{ifu} NAME"
                tid = str(hdr0.get(key_hdr)
                           or hdr0.get(f"HIERARCH {key_hdr}") or "").strip()
                if not tid or (field_hdr, tid) not in _Z_TABLE:
                    n_skipped += 1; continue
                group_id = ensure_group(field_hdr, tid)
            n_matched += 1

            # record meta
            if group_id not in group_to_meta:
                key_hdr = f"ESO OCS ARM{ifu} NAME"
                tid = str(hdr0.get(key_hdr)
                           or hdr0.get(f"HIERARCH {key_hdr}") or "").strip()
                group_to_meta[group_id] = (
                    field_hdr, tid,
                    _Z_TABLE.get((field_hdr, tid), np.nan),
                    _Z_TYPE.get((field_hdr, tid), ""),
                    _Z_QUALITY.get((field_hdr, tid)),
                )

            key = (group_id, band)
            g   = group_data[key]
            g["obs"].add(ob)

            try:
                cube = np.array(hdul[ifu].data, dtype=float)
            except Exception:
                continue
            hdr  = hdul[ifu].header
            wave = wavelength_axis(hdr)
            if g["wave"] is None:
                g["wave"] = wave

            _, _, z, *_ = group_to_meta.get(group_id, (None, None, np.nan, "", None))
            if np.isfinite(z):
                _, _, line_obs = pick_line(band, z)
                ctr = line_center(cube, wave, line_obs) or ifu_center(cube)
            else:
                line_obs = None
                ctr = ifu_center(cube)

            spec  = aperture_from_cube(cube, ctr)
            s_ref = np.nanmedian(spec)
            if np.isfinite(s_ref) and s_ref != 0:
                spec = spec / s_ref
            g["spectra"].append(spec)

            snr = line_snr(wave, spec, line_obs) if line_obs is not None else np.nan
            if np.isfinite(snr) and snr > g["best_snr"]:
                g["best_snr"] = snr
                g["best_cube"]   = cube
                g["best_center"] = ctr

# stack each group
all_stacked = []
for (group_id, band), g in group_data.items():
    if not g["spectra"]:
        continue
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        med = np.nanmedian(np.vstack(g["spectra"]), axis=0)
    field, tid, z, z_type, z_qual = group_to_meta.get(
        group_id, ("?", "?", np.nan, "", None))
    if np.isfinite(z):
        _, _, line_obs_f = pick_line(band, z)
        snr_u = line_snr(g["wave"], med, line_obs_f)
    else:
        snr_u = np.nan
    all_stacked.append({
        "group_id": group_id, "band": band,
        "field": field, "tid": tid,
        "z": z, "z_type": z_type, "z_qual": z_qual,
        "wave": g["wave"], "spec": med,
        "best_cube": g["best_cube"], "best_center": g["best_center"],
        "n_obs": len(g["obs"]),
        "snr_user": snr_u,
    })

all_stacked.sort(key=lambda e: -(e["snr_user"] if np.isfinite(e["snr_user"]) else -np.inf))
print(f"Loaded: {n_matched} IFU×OB matched, {n_skipped} skipped")
print(f"Stacked {len(all_stacked)} galaxy×band combinations")


In [ ]:
# ── match user stack ↔ external cube (spec-z only) ────────────────────────────

def extract_ext_cube(path):
    """Return (wave, data_cube, stat_cube) from DATACUBE_COMBINED."""
    with fits.open(path) as hdul:
        hdr  = hdul["DATA"].header
        cube = np.array(hdul["DATA"].data, dtype=float)
        stat = np.array(hdul["STAT"].data, dtype=float)
    return wavelength_axis(hdr), cube, stat

def pick_center(ctr_u, ctr_e, cube_u, cube_e, tol=1):
    """Line-peak center for both if they agree within tol px, else IFU centre."""
    if (ctr_u is not None and ctr_e is not None and
            abs(ctr_u[0] - ctr_e[0]) <= tol and abs(ctr_u[1] - ctr_e[1]) <= tol):
        return ctr_u, ctr_e, f"line peak {ctr_u}"
    return ifu_center(cube_u), ifu_center(cube_e), "IFU centre"

def normalise(wave, spec, ref_wl):
    cont = (np.abs(wave - (ref_wl - 3 * SNR_WIN_HALF)) <= CONT_HALF)
    ref  = np.nanmedian(spec[cont]) if cont.any() else np.nanmedian(spec)
    return spec / ref if (np.isfinite(ref) and ref != 0) else spec.copy()

def show_map(ax, img, title, ctr=None):
    if img is None or not np.isfinite(img).any():
        ax.text(0.5, 0.5, "No data", ha="center", va="center",
                transform=ax.transAxes, fontsize=9)
        ax.set_title(title, fontsize=9); ax.axis("off"); return
    vmin, vmax = np.nanpercentile(img, [2, 99])
    ax.imshow(img, origin="lower", vmin=vmin, vmax=vmax,
              cmap="inferno", aspect="equal", interpolation="nearest")
    if ctr is not None:
        ax.plot(ctr[1], ctr[0], "+", color="cyan", ms=8, mew=1.5)
    ax.set_title(title, fontsize=9); ax.axis("off")

# find matches
matched = []
for entry in all_stacked:
    if entry.get("z_type") != "spec":
        continue
    k = (entry["field"], str(entry["tid"]))
    if k in ext_index:
        matched.append((entry, ext_index[k]))

print(f"Spec-z galaxies matched in both datasets: {len(matched)}")

# ── pre-compute all results ────────────────────────────────────────────────────
all_results   = []
rows_summary  = [["field", "tid", "z", "band", "line", "N_OBs",
                  "SNR user", "SNR ext", "aperture"]]

for entry, ext_info in matched:
    z    = entry["z"]
    band = entry["band"]
    cube_u = entry["best_cube"]

    if cube_u is None:
        all_results.append(None)
        rows_summary.append([entry["field"], str(entry["tid"]), f"{z:.4f}",
                              band, "—", str(entry["n_obs"]), "—", "—", "—"])
        continue

    w_u = entry["wave"]
    _, line_name, line_obs = pick_line(band, z)

    try:
        w_e, cube_e, stat_e = extract_ext_cube(ext_info["path"])
    except Exception as exc:
        print(f"  skip {entry['field']}/{entry['tid']}: {exc}")
        all_results.append(None)
        rows_summary.append([entry["field"], str(entry["tid"]), f"{z:.4f}",
                              band, line_name, str(entry["n_obs"]), "—", "err", "—"])
        continue

    # aperture centre: line peak if both cubes agree within 1 px, else IFU centre
    ctr_u_lp = line_center(cube_u, w_u, line_obs) if np.isfinite(z) else None
    ctr_u    = ctr_u_lp if ctr_u_lp is not None else ifu_center(cube_u)
    ctr_e_lp = line_center(cube_e, w_e, line_obs) if np.isfinite(z) else None
    ctr_e    = ctr_e_lp if ctr_e_lp is not None else ifu_center(cube_e)
    ctr_u_f, ctr_e_f, ctr_mode = pick_center(ctr_u, ctr_e, cube_u, cube_e)

    # extract at the decided centre — same aperture/margin for both
    s_u_raw = aperture_from_cube(cube_u, ctr_u_f)
    s_e_raw = aperture_from_cube(cube_e, ctr_e_f)

    s_un = normalise(w_u, s_u_raw, line_obs)
    s_en = normalise(w_e, s_e_raw, line_obs)

    snr_u = line_snr(w_u, s_un, line_obs)
    snr_e = line_snr(w_e, s_en, line_obs)

    img_u = line_image(cube_u, w_u, line_obs)
    img_e = line_image(cube_e, w_e, line_obs)

    all_results.append({
        "entry": entry, "ext_info": ext_info,
        "w_u": w_u, "s_un": s_un,
        "w_e": w_e, "s_en": s_en,
        "snr_u": snr_u, "snr_e": snr_e,
        "line_name": line_name, "line_obs": line_obs,
        "ctr_u_f": ctr_u_f, "ctr_e_f": ctr_e_f, "ctr_mode": ctr_mode,
        "img_u": img_u, "img_e": img_e,
    })
    rows_summary.append([
        entry["field"], str(entry["tid"]), f"{z:.4f}",
        band, line_name, str(entry["n_obs"]),
        f"{snr_u:.1f}" if np.isfinite(snr_u) else "—",
        f"{snr_e:.1f}" if np.isfinite(snr_e) else "—",
        ctr_mode,
    ])

n_good = sum(1 for r in all_results if r is not None)
print(f"Computed {n_good} galaxy comparisons.")

# ── produce PDF ───────────────────────────────────────────────────────────────
with mpdf.PdfPages(OUT_PDF) as pdf:

    # -- summary page --
    fig_s, ax_s = plt.subplots(figsize=(16, 0.35 * max(len(matched), 1) + 2))
    ax_s.axis("off")
    tbl = ax_s.table(cellText=rows_summary[1:], colLabels=rows_summary[0],
                     loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8)
    tbl.auto_set_column_width(list(range(len(rows_summary[0]))))
    fig_s.suptitle("User best OB vs. External combined — SNR summary", fontweight="bold")
    pdf.savefig(fig_s, bbox_inches="tight"); plt.close(fig_s)

    # -- per-galaxy pages --
    for r in all_results:
        if r is None:
            continue
        entry     = r["entry"]
        field     = entry["field"]; tid = str(entry["tid"])
        z         = entry["z"];     band = entry["band"]
        line_name = r["line_name"]; line_obs = r["line_obs"]
        w_u, s_un = r["w_u"], r["s_un"]
        w_e, s_en = r["w_e"], r["s_en"]
        snr_u, snr_e     = r["snr_u"], r["snr_e"]
        ctr_u_f, ctr_e_f = r["ctr_u_f"], r["ctr_e_f"]
        ctr_mode         = r["ctr_mode"]
        img_u, img_e     = r["img_u"], r["img_e"]

        fig = plt.figure(figsize=(14, 9))
        fig.suptitle(
            f"{field}  |  target {tid}  |  z={z:.4f}  |  band={band}"
            f"  |  line={line_name} @ {line_obs:.4f} um"
            f"  |  N_OBs={entry['n_obs']}  |  t_ext={r['ext_info']['exptime']:.0f} s"
            f"  |  aperture: {ctr_mode}",
            fontsize=9, fontweight="bold",
        )
        gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)
        ax_full = fig.add_subplot(gs[0, :])
        ax_zoom = fig.add_subplot(gs[1, :2])
        ax_mu   = fig.add_subplot(gs[1, 2])
        ax_me   = fig.add_subplot(gs[1, 3])

        # full spectrum
        ax_full.plot(w_e, s_en, color="C0", lw=0.8, alpha=0.8,
                     label=(f"External  SNR={snr_e:.1f}"
                            if np.isfinite(snr_e) else "External  SNR=—"))
        inr = (w_u >= w_e[0]) & (w_u <= w_e[-1])
        ax_full.plot(w_u[inr], s_un[inr], color="C1", lw=0.8, alpha=0.8,
                     label=(f"User best OB  SNR={snr_u:.1f}"
                            if np.isfinite(snr_u) else "User best OB  SNR=—"))
        ax_full.axvline(line_obs, color="red", lw=0.7, ls="--", alpha=0.5,
                        label=f"{line_name} {line_obs:.4f} um")
        ylo, yhi = np.nanpercentile(s_en, [1, 99])
        ax_full.set_ylim(min(-0.5, ylo * 1.5), max(3.5, yhi * 1.5))
        ax_full.set_xlabel("wavelength (um)"); ax_full.set_ylabel("Normalised flux")
        ax_full.legend(fontsize=8, loc="upper right")
        ax_full.set_title(f"Full spectrum (normalised) — aperture: {ctr_mode}")

        # line zoom
        zm_e = (w_e >= line_obs - ZOOM_HALF) & (w_e <= line_obs + ZOOM_HALF)
        zm_u = (w_u >= line_obs - ZOOM_HALF) & (w_u <= line_obs + ZOOM_HALF)
        ax_zoom.plot(w_e[zm_e], s_en[zm_e], color="C0", lw=1.2, label="External")
        ax_zoom.plot(w_u[zm_u], s_un[zm_u], color="C1", lw=1.2, label="User best OB")
        ax_zoom.axvline(line_obs, color="red", lw=0.8, ls="--", alpha=0.7)
        ax_zoom.fill_betweenx([-5, 20], line_obs - SNR_WIN_HALF, line_obs + SNR_WIN_HALF,
                               alpha=0.08, color="red")
        ax_zoom.set_xlim(line_obs - ZOOM_HALF, line_obs + ZOOM_HALF)
        ax_zoom.set_xlabel("wavelength (um)"); ax_zoom.set_ylabel("Normalised flux")
        ax_zoom.legend(fontsize=8); ax_zoom.set_title(f"{line_name} zoom")

        show_map(ax_mu, img_u, f"User best OB ({line_name} collapsed)", ctr=ctr_u_f)
        show_map(ax_me, img_e, f"External combined ({line_name} collapsed)", ctr=ctr_e_f)

        plt.tight_layout(rect=[0, 0, 1, 0.94])
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)
        print(f"  {field}/{tid}  {line_name}  SNR user={snr_u:.1f}  ext={snr_e:.1f}  ctr={ctr_mode}")

print(f"\nSaved to {OUT_PDF}  ({n_good} galaxies)")
